### 1. Environment Setup & Authentication
Before manipulating data, we need to set up our toolkit. This section imports the required libraries (Pandas, Requests) and securely loads our Steam API key from the local `.env` file to ensure credentials are never hardcoded.

In [22]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

# On remonte d'un dossier pour trouver le fichier .env
load_dotenv('../.env')

STEAM_KEY = os.getenv("STEAM_API_KEY")

if STEAM_KEY:
    print("✅ Clé Steam chargée avec succès !")
else:
    print("❌ Erreur : Clé Steam introuvable. Vérifie ton fichier .env")

✅ Clé Steam chargée avec succès !


### 2. The Steam API Engine (Data Ingestion)
Here, we define the core `get_game_details` function. This acts as our direct line to the Steam public API. It takes a game's unique App ID and extracts the essential metadata required for our recommendation engine: name, clean description, genres, and pricing.

In [23]:
def get_game_details(app_id):
    """Récupère les détails d'un jeu via l'API publique de Steam"""
    url = f"http://store.steampowered.com/api/appdetails?appids={app_id}&l=french"
    response = requests.get(url).json()
    
    # Vérification que le jeu existe et a bien été trouvé
    if response and str(app_id) in response and response[str(app_id)]['success']:
        data = response[str(app_id)]['data']
        return {
            "id": app_id,
            "name": data.get('name', 'Inconnu'),
            "description": data.get('short_description', ''),
            "genres": [g['description'] for g in data.get('genres', [])],
            "price": data.get('price_overview', {}).get('final_formatted', 'Gratuit')
        }
    return None

print("✅ Fonction get_game_details prête !")

✅ Fonction get_game_details prête !


In [24]:
import os

# Liste le contenu du dossier data
data_path = "data" # ou "/app/data"
if os.path.exists(data_path):
    files = os.listdir(data_path)
    print(f"✅ Dossier '{data_path}' accessible.")
    print(f"📄 Contenu : {files}")
    if "games.csv" in files:
        print("🎯 C'EST GAGNÉ : Le fichier games.csv est enfin là !")
    else:
        print("⚠️ Le dossier est là, mais games.csv manque à l'appel.")
else:
    print("❌ Dossier data introuvable.")

✅ Dossier 'data' accessible.
📄 Contenu : ['chroma_db', 'games.csv']
🎯 C'EST GAGNÉ : Le fichier games.csv est enfin là !


### 3. Data Scraping & Preprocessing (Mining Hidden Gems)
Instead of hardcoding popular games, we use the SteamSpy API to dynamically fetch games based on specific thematic tags (e.g., "Space", "Cyberpunk"). 
We gather the details using our Steam function, implement a slight delay to respect API rate limits, and perform **Data Cleaning** to strip out unwanted HTML tags. The output is a structured Pandas DataFrame containing the unified text (`combined_content`) ready for vectorization.

In [25]:
import pandas as pd
import re

file_path = "/app/data/games.csv"

print("⏳ Lecture du CSV et correction du décalage...")
df_raw = pd.read_csv(file_path, low_memory=False)

# 🔧 LE CORRECTIF DU DÉCALAGE KAGGLE 🔧
# On force Pandas à recracher l'index (qui contient les vrais AppID) en tant que colonne
df_raw = df_raw.reset_index() 

# On renomme les colonnes décalées pour récupérer nos vraies données
df_raw.rename(columns={
    'index': 'Vrai_AppID', 
    'AppID': 'Vrai_Name', 
    'Required age': 'Vrai_Price' # Le prix avait glissé ici !
}, inplace=True)

# On sélectionne nos colonnes corrigées + celles qui étaient bien alignées (Tags, About)
cols = ['Vrai_AppID', 'Vrai_Name', 'About the game', 'Tags', 'Vrai_Price']
df = df_raw[cols].copy()

# On nettoie le nom des colonnes pour la suite de ton code
df.rename(columns={
    'Vrai_AppID': 'AppID', 
    'Vrai_Name': 'Name', 
    'Vrai_Price': 'Price'
}, inplace=True)

# 🧹 Nettoyage et échantillonnage
df = df.dropna(subset=['About the game', 'Name'])
df_labo = df.sample(n=2000, random_state=42).copy()

def clean_description(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'<.*?>', '', text)
    return re.sub(r'\s+', ' ', text).strip()

df_labo['combined_content'] = df_labo.apply(
    lambda row: f"NOM DU JEU: {row['Name']}. TAGS: {row['Tags']}. DESCRIPTION: {clean_description(row['About the game'])}", 
    axis=1
)

print(f"✅ Dataset corrigé et chargé ! {len(df_labo)} jeux prêts.")
display(df_labo[['Name', 'Tags', 'Price']].head())

⏳ Lecture du CSV et correction du décalage...
✅ Dataset corrigé et chargé ! 2000 jeux prêts.


,Name,Tags,Price
98815,Mushroom: The Ruckus,"Action,Indie,Casual",0.55
62661,Train of Thoughts,"Choose Your Own Adventure,Typing,Word Game,PvP...",1.99
114549,They See Us,NaN,2.99
104995,Brutal Legend,"Action,Comedy,Adventure,Great Soundtrack,Open ...",2.24
65078,Machine World 2,"Simulation,Early Access,Indie,Sandbox,Realisti...",12.99


### 4. Vector Database Initialization (ChromaDB)
This step initializes our local memory. We instantiate **ChromaDB** to store our game data. More importantly, we load the Open Source `all-MiniLM-L6-v2` embedding model. This model will act as the translator, converting our human-readable game descriptions into mathematical vectors.

In [26]:
import chromadb
from chromadb.utils import embedding_functions

# On crée la base de données à la racine du projet (../data/chroma_db)
chroma_client = chromadb.PersistentClient(path="../data/chroma_db")

# On utilise un modèle open-source léger pour transformer le texte en vecteurs
# La première fois, ça va télécharger le modèle (environ 80 Mo)
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Création (ou récupération) de la "boîte" où on va ranger nos jeux
collection = chroma_client.get_or_create_collection(
    name="gamepulse_collection",
    embedding_function=sentence_transformer_ef
)

print(f"✅ ChromaDB est prêt ! La collection contient actuellement {collection.count()} éléments.")

✅ ChromaDB est prêt ! La collection contient actuellement 2000 éléments.


### 5. Populating the Vector Store
With our database initialized and our dataset cleaned, we now map the data into ChromaDB. We extract the raw text, the unique game IDs, and the metadata (so we can display human-readable names and prices later). The embedding function automatically vectorizes the text during the insertion process.

In [27]:
# Préparation des données depuis ton dataset Kaggle
documents = df_labo['combined_content'].tolist()
ids = df_labo['AppID'].astype(str).tolist()
metadatas = df_labo[['Name', 'Price']].to_dict(orient='records')

# On vide la collection avant pour éviter les mélanges avec tes anciens tests
try:
    existing_ids = collection.get()['ids']
    if existing_ids:
        collection.delete(ids=existing_ids)
        print("🧹 Ancienne collection vidée.")
except:
    pass

# Insertion massive
collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas
)

print(f"✅ Succès ! {len(documents)} jeux ont été vectorisés et stockés dans ChromaDB.")

🧹 Ancienne collection vidée.
✅ Succès ! 2000 jeux ont été vectorisés et stockés dans ChromaDB.


### 6. Semantic Search & Retrieval (RAG Testing)
The final step of the pipeline. We simulate a user query in natural language. ChromaDB calculates the semantic distance between the user's prompt and the vector space of our stored games. It then returns the closest matches, proving that the system understands the *context* and *theme* of the request, rather than just relying on exact keyword matching.

In [31]:
# 1. Ta question (tu peux la changer à l'infini !)
query = "Je cherche un jeu de puzzle cozy et mignon, idéalement avec des chats"

# 2. La recherche dans ChromaDB
results = collection.query(
    query_texts=[query],
    n_results=3 # On demande les 3 meilleurs résultats
)

# 3. Affichage des résultats
print(f"🔍 Résultats pour : '{query}'\n")

for i in range(len(results['ids'][0])):
    # Attention aux majuscules 'Name' et 'Price' qui correspondent à nos nouvelles métadonnées
    game_name = results['metadatas'][0][i]['Name']
    game_price = results['metadatas'][0][i]['Price']
    game_desc = results['documents'][0][i][:200] # On affiche les 200 premiers caractères
    
    print(f"🎮 Jeu trouvé : {game_name}")
    print(f"💰 Prix : {game_price}")
    print(f"📝 Extrait : {game_desc}...")
    print("-" * 40)

🔍 Résultats pour : 'Je cherche un jeu de puzzle cozy et mignon, idéalement avec des chats'

🎮 Jeu trouvé : PBN  - Da Game
💰 Prix : 2.0
📝 Extrait : NOM DU JEU: PBN  - Da Game. TAGS: Adventure,2D Fighter,Arcade,Farming Sim,Interactive Fiction,2D Platformer,Top-Down Shooter,2D,Pixel Graphics,Indie,Romance,Combat,Story Rich,Singleplayer. DESCRIPTION...
----------------------------------------
🎮 Jeu trouvé : Cozy Halloween
💰 Prix : 0.49
📝 Extrait : NOM DU JEU: Cozy Halloween. TAGS: nan. DESCRIPTION: Cozy Halloween is an engaging puzzle game, position the pieces correctly inside the shelf. On a rainy day, make some hot chocolate, snuggle up and s...
----------------------------------------
🎮 Jeu trouvé : The little Puzzle House
💰 Prix : 2.49
📝 Extrait : NOM DU JEU: The little Puzzle House. TAGS: Casual,Point & Click,Puzzle,3D,Logic,Singleplayer,Indie. DESCRIPTION: Enjoy a relaxing puzzle game and unlock rooms in the house to progress. Find objects fo...
--------------------------------------